<a href="https://colab.research.google.com/github/lopezale95/AA2/blob/main/Regresion_Contexto3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de regresión lineal simple y múltiple – Contexto 3
**Salario diario de operarios** en función de las horas de trabajo, el área terminada (m²), los minutos de retraso y el material reciclado (kg).

Ejecuta las celdas en orden (menú *Entorno de ejecución → Ejecutar todas*). Al final se descarga un ZIP con las tablas en Excel y las figuras.

## 0. Instalación e importación de librerías

In [ ]:
# Colab ya trae estas librerías; la línea asegura que estén instaladas
!pip install -q numpy pandas scipy matplotlib openpyxl

import itertools
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
print("Librerías cargadas")

## 1. Datos (Tabla 3 del enunciado)

In [ ]:
d = pd.DataFrame({
    "Horas":     [7, 6, 6, 4, 8, 6, 7, 6, 9],
    "Area":      [43, 39, 34, 31, 28, 34, 37, 37, 29],
    "Retraso":   [0, 17, 1, 60, 22, 4, 10, 26, 0],
    "Reciclado": [6, 4, 7, 3, 4, 3, 5, 5, 4],
    "Salario":   [64200, 50000, 54000, 20000, 41000, 49000, 54000, 42000, 55000],
}, index=range(1, 10))
d.index.name = "Operario"
X = ["Horas", "Area", "Retraso", "Reciclado"]
y = d["Salario"].to_numpy(float)
n = len(d)
d

NameError: name 'pd' is not defined

In [ ]:
# Estadísticos descriptivos
descriptivos = d.describe().T
descriptivos

## 2. Función de regresión por mínimos cuadrados ordinarios
$\hat\beta = (X'X)^{-1}X'y$

In [ ]:
def ols(cols):
    Xm = np.column_stack([np.ones(n)] + [d[c].to_numpy(float) for c in cols])
    p = Xm.shape[1]; k = p - 1
    XtX_inv = np.linalg.inv(Xm.T @ Xm)
    b = XtX_inv @ Xm.T @ y                      # coeficientes
    y_hat = Xm @ b
    e = y - y_hat                               # residuos
    sce = e @ e                                 # SC error
    sct = ((y - y.mean()) ** 2).sum()           # SC total
    scr = sct - sce                             # SC regresión
    gl_e = n - p
    cme = sce / gl_e
    se = np.sqrt(np.diag(XtX_inv) * cme)
    t = b / se
    t_c = stats.t.ppf(0.975, gl_e)
    F = (scr / k) / cme
    return {"cols": cols, "b": b, "se": se, "t": t, "p": 2 * stats.t.sf(np.abs(t), gl_e),
            "ic_inf": b - t_c * se, "ic_sup": b + t_c * se,
            "R2": scr / sct, "R2_aj": 1 - (sce / gl_e) / (sct / (n - 1)),
            "F": F, "p_F": stats.f.sf(F, k, gl_e), "S": np.sqrt(cme),
            "SCT": sct, "SCR": scr, "SCE": sce, "gl_e": gl_e, "k": k,
            "y_hat": y_hat, "e": e, "H": Xm @ XtX_inv @ Xm.T, "Xm": Xm}

abrev = {"Horas": "H", "Area": "A", "Retraso": "R", "Reciclado": "K"}

def ecuacion(m):
    return f"Ŷ = {m['b'][0]:,.2f}" + "".join(
        f" {'-' if bj < 0 else '+'} {abs(bj):,.2f}·{abrev[c]}" for bj, c in zip(m["b"][1:], m["cols"]))

def tabla_modelos(lista):
    filas = []
    for m in lista:
        filas.append({"Variables": " + ".join(abrev[c] for c in m["cols"]),
                      "Ecuación": ecuacion(m), "R2": m["R2"], "R2_ajustado": m["R2_aj"],
                      "S": m["S"], "F": m["F"], "p_F": m["p_F"],
                      "Coef. signif. 5%": ", ".join(abrev[c] for c, pv in zip(m["cols"], m["p"][1:]) if pv < 0.05) or "ninguno"})
    return pd.DataFrame(filas)

# Se estiman los 15 modelos posibles
modelos = {comb: ols(list(comb)) for k in range(1, 5) for comb in itertools.combinations(X, k)}
print(f"{len(modelos)} modelos estimados")

## 3. Matriz de correlación de Pearson

In [ ]:
correlacion = d.corr()
correlacion

In [ ]:
# Orden de las variables según |r| con el salario
orden = correlacion["Salario"].drop("Salario").sort_values(key=abs, ascending=False)
orden.to_frame("r con Salario")

## 4. Regresión lineal simple (pregunta 2)

In [ ]:
simples = tabla_modelos([modelos[(c,)] for c in X])
simples

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(9, 6.5))
etiquetas = {"Horas": "Horas de trabajo", "Area": "Área terminada (m²)",
             "Retraso": "Minutos de retraso", "Reciclado": "Material reciclado (kg)"}
for a, c in zip(ax.flat, X):
    m = modelos[(c,)]
    a.scatter(d[c], y / 1000, color="#2b6cb0", s=30, zorder=3)
    xs = np.linspace(d[c].min(), d[c].max(), 50)
    a.plot(xs, (m["b"][0] + m["b"][1] * xs) / 1000, color="#8a8a8a")
    a.set_xlabel(etiquetas[c]); a.set_ylabel("Salario (miles $)")
    a.set_title(f"r = {correlacion.loc[c, 'Salario']:.3f}   R² = {m['R2']:.3f}", loc="left")
fig.suptitle("Figura 1. Salario vs. cada variable con recta de MCO")
fig.tight_layout(); fig.savefig("fig1_dispersion.png", dpi=200); plt.show()

## 5. Regresión múltiple con dos variables (pregunta 3)

In [ ]:
dos = tabla_modelos([m for c, m in modelos.items() if len(c) == 2])
dos.sort_values("R2", ascending=False)

## 6. Regresión múltiple con tres variables (pregunta 4)

In [ ]:
tres = tabla_modelos([m for c, m in modelos.items() if len(c) == 3])
tres.sort_values("R2", ascending=False)

In [ ]:
todos = tabla_modelos(list(modelos.values()))
fig, a = plt.subplots(figsize=(9, 4))
pos = range(len(todos))
a.bar(pos, todos["R2"], color="#2b6cb0", width=0.7, label="R²")
a.scatter(pos, todos["R2_ajustado"].clip(lower=0), marker="_", s=200, color="#c0392b", zorder=3, label="R² ajustado")
a.set_xticks(list(pos)); a.set_xticklabels(todos["Variables"].str.replace(" ", ""), rotation=45)
a.set_ylabel("R²"); a.set_ylim(0, 1.05); a.legend(frameon=False)
a.set_title("Figura 2. R² y R² ajustado de los 15 modelos")
fig.tight_layout(); fig.savefig("fig2_r2_modelos.png", dpi=200); plt.show()

## 7. Modelo completo y pruebas de hipótesis (pregunta 5)

In [ ]:
full = modelos[tuple(X)]
print("Modelo completo:", ecuacion(full))
ajuste = pd.DataFrame({"Valor": [full["R2"], full["R2_aj"], full["S"]]},
                      index=["R²", "R² ajustado", "Error estándar S"])
ajuste

In [ ]:
# Prueba global (ANOVA)  H0: b1 = b2 = b3 = b4 = 0
anova = pd.DataFrame({
    "gl": [full["k"], full["gl_e"], n - 1],
    "SC": [full["SCR"], full["SCE"], full["SCT"]],
    "CM": [full["SCR"] / full["k"], full["SCE"] / full["gl_e"], np.nan],
    "F":  [full["F"], np.nan, np.nan],
    "Valor p": [full["p_F"], np.nan, np.nan]}, index=["Regresión", "Residuo", "Total"])
print(f"F crítico (α=0.05; 4, 4) = {stats.f.ppf(0.95, full['k'], full['gl_e']):.3f}")
anova

In [ ]:
# Pruebas parciales (t) e intervalos de confianza al 95 %   H0: bj = 0
coeficientes = pd.DataFrame({
    "Coeficiente": full["b"], "Error estándar": full["se"], "t": full["t"], "Valor p": full["p"],
    "IC 95% inf": full["ic_inf"], "IC 95% sup": full["ic_sup"]}, index=["Intercepto"] + X)
coeficientes["Decisión (α=0.05)"] = np.where(coeficientes["Valor p"] < 0.05, "Rechaza H0", "No rechaza H0")
coeficientes.loc["Intercepto", "Decisión (α=0.05)"] = "—"
print(f"t crítico (0.025; {full['gl_e']}) = {stats.t.ppf(0.975, full['gl_e']):.3f}")
coeficientes

## 8. Evaluación de supuestos

In [ ]:
e = full["e"]
sw = stats.shapiro(e)                                       # normalidad
dw = np.sum(np.diff(e) ** 2) / np.sum(e ** 2)               # independencia
# Breusch-Pagan: LM = n·R² de la regresión auxiliar de e² sobre los regresores
g = e ** 2; Z = full["Xm"]
bz = np.linalg.lstsq(Z, g, rcond=None)[0]
lm = n * (1 - ((g - Z @ bz) ** 2).sum() / ((g - g.mean()) ** 2).sum())
p_lm = stats.chi2.sf(lm, len(X))
# VIF (multicolinealidad)
vif = {}
for c in X:
    otras = [x for x in X if x != c]
    Xo = np.column_stack([np.ones(n)] + [d[x] for x in otras])
    bo = np.linalg.lstsq(Xo, d[c], rcond=None)[0]
    vif[c] = 1 / (((d[c] - Xo @ bo) ** 2).sum() / ((d[c] - d[c].mean()) ** 2).sum())

supuestos = pd.DataFrame({
    "Prueba": ["Shapiro-Wilk", "Breusch-Pagan (LM)", "Durbin-Watson", "VIF"],
    "Estadístico": [f"W = {sw.statistic:.4f}", f"LM = {lm:.4f}", f"d = {dw:.4f}",
                    ", ".join(f"{abrev[c]} = {v:.2f}" for c, v in vif.items())],
    "Valor p": [round(sw.pvalue, 4), round(p_lm, 4), "—", "—"]},
    index=["Normalidad", "Homocedasticidad", "Independencia", "Multicolinealidad"])
supuestos

In [ ]:
diagnostico = pd.DataFrame({"Salario": y, "Ajustado": full["y_hat"], "Residuo": e,
                            "Leverage h": np.diag(full["H"])}, index=d.index)
diagnostico

In [ ]:
lev = diagnostico["Leverage h"].to_numpy()
r_est = e / (full["S"] * np.sqrt(1 - lev))
fig, ax = plt.subplots(1, 3, figsize=(11, 3.4))
ax[0].scatter(full["y_hat"] / 1000, e, color="#2b6cb0"); ax[0].axhline(0, color="#8a8a8a")
for i, (xv, yv) in enumerate(zip(full["y_hat"] / 1000, e), start=1):
    ax[0].annotate(str(i), (xv, yv), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax[0].set_xlabel("Ajustados (miles $)"); ax[0].set_ylabel("Residuo ($)"); ax[0].set_title("Residuos vs. ajustados")
(osm, osr), (pend, interc, _) = stats.probplot(r_est)
ax[1].scatter(osm, osr, color="#2b6cb0"); ax[1].plot(osm, pend * osm + interc, color="#8a8a8a")
ax[1].set_xlabel("Cuantiles teóricos"); ax[1].set_ylabel("Residuo estudentizado"); ax[1].set_title("Q-Q normal")
ax[2].bar(range(1, n + 1), lev, color=["#c0392b" if h > 0.8 else "#2b6cb0" for h in lev])
ax[2].set_xticks(range(1, n + 1)); ax[2].set_xlabel("Operario"); ax[2].set_ylabel("Leverage h"); ax[2].set_title("Apalancamiento")
fig.suptitle("Figura 3. Diagnóstico de residuos del modelo completo")
fig.tight_layout(); fig.savefig("fig3_diagnostico.png", dpi=200); plt.show()

## 9. Efecto de cada variable (pregunta 6)

In [ ]:
for c, txt in zip(X, ["cada hora de trabajo", "cada m² terminado", "cada minuto de retraso", "cada kg reciclado"]):
    i = X.index(c) + 1
    print(f"{txt:<24}: {full['b'][i]:>10,.2f} $   IC 95%: [{full['ic_inf'][i]:,.2f} ; {full['ic_sup'][i]:,.2f}]")

## 10. Exportar resultados (Excel + figuras en un ZIP)

In [ ]:
import zipfile
with pd.ExcelWriter("resultados_regresion_contexto3.xlsx") as w:
    d.to_excel(w, sheet_name="Datos")
    descriptivos.to_excel(w, sheet_name="Descriptivos")
    correlacion.to_excel(w, sheet_name="Correlacion")
    simples.to_excel(w, sheet_name="Simples", index=False)
    dos.to_excel(w, sheet_name="Dos variables", index=False)
    tres.to_excel(w, sheet_name="Tres variables", index=False)
    anova.to_excel(w, sheet_name="ANOVA")
    coeficientes.to_excel(w, sheet_name="Coeficientes")
    supuestos.to_excel(w, sheet_name="Supuestos")
    diagnostico.to_excel(w, sheet_name="Residuos")

archivos = ["resultados_regresion_contexto3.xlsx", "fig1_dispersion.png", "fig2_r2_modelos.png", "fig3_diagnostico.png"]
with zipfile.ZipFile("resultados_contexto3.zip", "w") as z:
    for a in archivos:
        z.write(a)
print("ZIP creado:", archivos)

try:                                    # descarga automática en Colab
    from google.colab import files
    files.download("resultados_contexto3.zip")
except ImportError:
    print("Fuera de Colab: el ZIP quedó en la carpeta actual")